# Step 4
Step 4 gives MedImageForge a reliable memory of its data: what exists, what changed, what disappeared, and what state each file is in




We created a database that keeps track of every file in our medical dataset.

Before Step 4:
```
data/
   ↓
lots of files
```
MedImageForge could inspect those files, but it didn't have a persistent inventory.

After Step 4:
```
data/
   ↓
actual images and metadata

        +
        
artifacts/manifest.db
   ↓
database describing every file
```
## 1. What did we build?

Two things:
``` python 
manifest.py
```
This contains the ingestion logic.

Its job is basically:

Find files → examine them → record them in the database.

ingest command

You can run:
``` python 
python -m medimageforge ingest
```
This tells MedImageForge:

"Go through my dataset and update the file inventory."

In [2]:
! python -m medimageforge ingest       


2026-09-14 14:50:12,630 | INFO    | medimageforge.cli | Ingesting /home/zahra/MedImageForge/data into /home/zahra/MedImageForge/artifacts/manifest.db
=== Ingest report ===
Files on disk:  5326
New:            0
Unchanged:      5326
Updated:        0
Missing:        0
Manifest:       /home/zahra/MedImageForge/artifacts/manifest.db


In [17]:
import sqlite3

db_path = r"../artifacts/manifest.db"

conn = sqlite3.connect(db_path)

count = conn.execute("SELECT COUNT(*) FROM files").fetchone()[0]

print("Number of records:", count)

conn.close()

Number of records: 5326







## 2. What does "fingerprint every file" mean?

For every file, we calculate:

SHA-256

Think of SHA-256 as a digital fingerprint for the file.

For example:
``` 
image A
   ↓
SHA-256
   ↓
ABC123...
```

If the content changes:
```
image A
   ↓
SHA-256
   ↓
XYZ789...
```
The fingerprint changes.

So MedImageForge can detect:

"This file is not the same as before."


## 3. What is the manifest?

The manifest is:
```
artifacts/manifest.db
```

It is a SQLite database.

Imagine it as a large table:

| File        | Patient | Type  | Hash   | Status |
| ----------- | ------- | ----- | ------ | ------ |
| brain/1.jpg | 049     | slice | ABC... | raw    |
| bone/1.jpg  | 049     | slice | DEF... | raw    |
| mask/1.jpg  | 049     | mask  | XYZ... | raw    |


So instead of asking the filesystem:

"What files do I have?"

we can ask the database:

"Show me all brain slices for patient 049."

That's much more powerful.


## 4. The most important part: what happens during ingestion?

This is the heart of Step 4.

For every file:

#### Case 1 — New file

Database doesn't know it.
```
file
 ↓
not in database
 ↓
INSERT
```


#### Case 2 — Same file

Database already knows it and:
```
SHA-256 = same
size    = same
```
So:

nothing fundamentally changed

We simply update last_seen.

Result:

unchanged



#### Case 3 — File changed

The path is the same, but:

old SHA-256 ≠ new SHA-256

So MedImageForge knows:

"Something changed inside this file."

It updates the database record.

Result:

updated


#### Case 4 — File disappeared

Suppose the database knows:

049/brain/17.jpg

but the file is no longer on disk.

We don't delete the database record.

Instead:

status = missing

This is important.

The history remains:
```
This file existed.
We registered it.
Later it disappeared.
```
That information can matter for auditability and reproducibility.